In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

# Load data
driver_standings = pd.read_csv('archive/cleaned_driver_standings.csv')




,driverId,driver_code,driver_forename,driver_surname,driver_dob,driver_nationality,driver_home
176,177,\N,Keke,Rosberg,1948-12-06,Finnish,Finland


In [3]:
print(races.columns)


Index(['raceId', 'year', 'round', 'circuitId'], dtype='object')


In [23]:
import statsmodels.api as sm

races = pd.read_csv('archive/cleaned_races.csv')
results = pd.read_csv('archive/cleaned_results.csv')
drivers = pd.read_csv('archive/cleaned_drivers.csv')
results = results.merge(races[['raceId', 'year']], on='raceId', how='left')
results = results.merge(drivers[['driverId', 'driver_dob']], on='driverId', how='left')

results['position_numeric'] = pd.to_numeric(results['position'], errors='coerce')

results['driver_dob'] = pd.to_datetime(results['driver_dob'])

results['age'] = results['year'] - results['driver_dob'].dt.year

results["prime"] = abs(results['age'] - 30)

results = results.dropna(subset=['position_numeric'])

results = results.sort_values(['driverId','year'])

def calculate_past_stats(group):
    wins = []
    avg_placement = []
    total_wins = 0
    placements = []
    for pos in group['position_numeric']:
        if placements:
            avg_placement.append(np.mean(placements))
            wins.append(total_wins)
        else:
            avg_placement.append(np.nan)
            wins.append(0)
        placements.append(pos)
        if pos == 1:
            total_wins += 1
    group['total_wins_before'] = wins
    group['avg_placement_before'] = avg_placement
    return group

results = results.groupby('driverId', group_keys=False).apply(calculate_past_stats).reset_index(drop=True)
results = results.dropna(subset=['avg_placement_before'])
results["placement_prime"] = results["avg_placement_before"] * results["prime"]
print(results)

       resultId  raceId  driverId  constructorId  grid_position  position  \
1           392      37         1              1              4         2   
2           414      38         1              1              2         2   
3           436      39         1              1              4         2   
4           458      40         1              1              2         2   
5           479      41         1              1              1         1   
...         ...     ...       ...            ...            ...       ...   
26075     26005    1106       858              3             18        20   
26076     26018    1107       858              3             18        13   
26077     26036    1108       858              3             14        11   
26078     26063    1109       858              3             20        18   
26079     26082    1110       858              3             18        17   

       points  laps  fastestLap  fastestLapRank  ... fastestLapSpeed  \
1  

/var/folders/xy/t1f6_zl1051c9h_2tjj691gr0000gn/T/ipykernel_77745/2044976417.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results = results.groupby('driverId', group_keys=False).apply(calculate_past_stats).reset_index(drop=True)


In [31]:


X = results[['avg_placement_before' ]]
y_linear = results['position_numeric']
y_logistic = (results['position_numeric'] == 1).astype(int)

X_ols = sm.add_constant(X)

ols_model = sm.OLS(y_linear, X_ols)
ols_results = ols_model.fit()

print("OLS Linear Regression Results:")
print(ols_results.summary())

logit_model = sm.Logit(y_logistic, X_ols)
logit_results = logit_model.fit()

print("Logistic Regression Results (OLS version):")
print(logit_results.summary())

# --- Logistic Regression using sklearn ---
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X, y_logistic, test_size=0.3)
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_log, y_train_log)
y_pred_log = logistic_model.predict(X_test_log)

print("Logistic Regression Results (sklearn):")
print("Accuracy:", accuracy_score(y_test_log, y_pred_log))
print("Coefficients:", logistic_model.coef_)
print("Intercept:", logistic_model.intercept_)


OLS Linear Regression Results:
                            OLS Regression Results                            
Dep. Variable:       position_numeric   R-squared:                       0.225
Model:                            OLS   Adj. R-squared:                  0.225
Method:                 Least Squares   F-statistic:                     7305.
Date:                Thu, 08 May 2025   Prob (F-statistic):               0.00
Time:                        16:08:12   Log-Likelihood:                -83897.
No. Observations:               25223   AIC:                         1.678e+05
Df Residuals:                   25221   BIC:                         1.678e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
c

In [27]:
results.to_csv('results.csv', index=False)